# Exploratory Data Analysis

## Alberta Electricity Price Predictor

This notebook explores the current historical Alberta electricity price dataset before feature engineering and modeling.

The goal is to understand the dataset from both a technical and business perspective. The analysis checks data coverage, data quality, price distribution, time-based patterns, spike behavior, and forecast usefulness.

These findings will guide later decisions about feature engineering, model design, spike-risk labels, and recommendation logic. No final `recommended`, `acceptable`, or `avoid` thresholds are defined in this notebook.

## Business Questions

This EDA is organized around 10 essential business questions.

The goal is not only to describe the dataset. The goal is to understand which patterns matter for household electricity decisions and which issues must be resolved before feature engineering, modeling, and recommendation logic.

1. What time period does the dataset cover?
2. Is the hourly time series complete and reliable?
3. How are actual electricity prices distributed?
4. How often are prices zero, low, high, or extreme?
5. When do price spikes occur?
6. What hours of the day tend to be cheaper or more expensive?
7. What day-of-week patterns exist?
8. What monthly or seasonal patterns exist?
9. How does the AESO forecast compare with actual prices?
10. What should be studied before defining recommendation thresholds?

## Imports

In [ ]:
# Path helps us create file paths that work across different operating systems.
from pathlib import Path

# Matplotlib is used to create simple charts for the EDA.
import matplotlib.pyplot as plt

# Pandas is used to load, clean, group, and summarize the dataset.
import pandas as pd

## Load Dataset

In [ ]:
# Store the path to the current historical dataset created by the data pipeline.
DATASET_PATH = Path("../data/interim/current_historical_prices_clean.csv")

# Load the current historical dataset into a pandas DataFrame.
df = pd.read_csv(DATASET_PATH)

# Convert the UTC timestamp column from text to datetime for time-series analysis.
df["datetime_universal_time"] = pd.to_datetime(df["datetime_universal_time"])

# Convert the Alberta local timestamp column from text to datetime for local time analysis.
df["datetime_local_time"] = pd.to_datetime(df["datetime_local_time"])

# Display the first rows to confirm the dataset loaded correctly.
df.head()

In [ ]:
# Print the number of rows in the dataset.
print(f"Rows: {df.shape[0]:,}")

# Print the number of columns in the dataset.
print(f"Columns: {df.shape[1]}")

# Print the first and last UTC timestamps to understand the dataset coverage.
print(
  f"Date range UTC: "
  f"{df['datetime_universal_time'].min()} to {df['datetime_universal_time'].max()}"
)

## 1. What time period does the dataset cover?

This question checks whether the dataset has enough historical coverage for time-based analysis and future machine learning.

In [ ]:
# Get the first UTC timestamp in the dataset.
start_time = df["datetime_universal_time"].min()

# Get the latest UTC timestamp in the dataset.
end_time = df["datetime_universal_time"].max()

# Calculate the number of full days covered by the dataset.
total_days = (end_time - start_time).days

# Convert days into approximate years for easier interpretation.
total_years = total_days / 365.25

# Print the first timestamp in the dataset.
print(f"Start UTC time: {start_time}")

# Print the latest timestamp in the dataset.
print(f"End UTC time: {end_time}")

# Print the total number of days covered.
print(f"Total days: {total_days:,}")

# Print the approximate number of years covered.
print(f"Approximate years: {total_years:.2f}")

### Finding

The dataset provides enough historical coverage for early modeling work.

It covers more than six years of hourly Alberta electricity price data, from January 2020 to July 2026. This gives the project enough history to study short-term patterns, such as hour-of-day and day-of-week effects, as well as longer-term patterns, such as monthly and seasonal changes.

This coverage is important because electricity prices are not stable over time. A multi-year dataset helps the project compare normal periods with more volatile periods before building forecasting and spike-risk models.

The dataset is suitable for EDA and feature engineering, but the modeling phase should still use time-based train, validation, and test splits to avoid data leakage.

## 2. Is the hourly time series complete and reliable?

This question checks whether the dataset has duplicate or missing hourly UTC timestamps.

In [ ]:
# Count duplicate UTC timestamps.
duplicate_count = df["datetime_universal_time"].duplicated().sum()

# Create the full expected hourly timeline between the first and latest UTC timestamp.
expected_hours = pd.date_range(
  start=df["datetime_universal_time"].min(),
  end=df["datetime_universal_time"].max(),
  freq="h",
)

# Compare the expected hourly timeline with the timestamps that exist in the dataset.
missing_hours = expected_hours.difference(df["datetime_universal_time"])

# Print the number of duplicate UTC timestamps.
print(f"Duplicate UTC timestamps: {duplicate_count}")

# Print the number of missing hourly timestamps.
print(f"Missing hourly UTC timestamps: {len(missing_hours)}")

### Finding

The hourly time series is complete and reliable.

The dataset has no duplicate UTC timestamps and no missing hourly UTC timestamps. This means each hour appears once, and the timeline is continuous from the first record to the latest record.

This is important because electricity price forecasting depends on time order. Missing hours or duplicate hours could distort hourly patterns, lag features, rolling averages, and model evaluation.

For this project, the dataset is reliable enough to continue with exploratory analysis by hour, day, month, and season.

## 3. How are actual electricity prices distributed?

This question helps identify the normal price range, unusual values, and whether electricity prices are stable or highly variable.

In [ ]:
# Summarize actual prices using standard statistics and selected percentiles.

actual_price_summary = df["actual_price"].describe(
  percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]

)

# Display the actual price summary.
actual_price_summary

In [ ]:
# Create a new chart area.
plt.figure(figsize=(10, 5))

# Plot the distribution of actual prices and remove missing values from the chart.
plt.hist(df["actual_price"].dropna(), bins=100)

# Add a clear chart title.
plt.title("Distribution of Actual Electricity Prices")

# Label the x-axis with the price unit.
plt.xlabel("Actual price ($/MWh)")

# Label the y-axis with the number of hourly records.
plt.ylabel("Number of hours")

# Display the chart.
plt.show()

### Finding

Actual electricity prices are highly skewed.

Most hours have low or moderate prices, but a smaller number of expensive hours pull the average upward. The median actual price is about 41 CAD/MWh, while the mean is about 88 CAD/MWh. This gap shows that the average price is strongly affected by high-price periods.

The upper end of the distribution is especially important. The 95th percentile is about 396 CAD/MWh, and the 99th percentile is about 837 CAD/MWh. This means that a small share of hours can be dramatically more expensive than a typical hour.

For the business problem, this matters because households do not only need to know the average price. They need help identifying when prices are unusually high and when flexible electricity use may be safer.

These findings support both parts of the project: price prediction and spike-risk modeling.

## 4. How often are prices zero, low, high, or extreme?

This question helps understand how often households may see unusually cheap or expensive electricity hours.

In [ ]:
# Keep only rows where actual_price is available.
valid_prices = df["actual_price"].dropna()

# Split actual prices into mutually exclusive business-friendly ranges.
price_bins = pd.cut(
  valid_prices,
  bins=[-0.01, 0, 25, 100, 250, 500, float("inf")],
  labels=[
    "zero",
    "0_to_25",
    "25_to_100",
    "100_to_250",
    "250_to_500",
    "500_plus",
  ],
)

# Count how many hourly records fall into each price range.
price_range_summary = (
  price_bins
  .value_counts()
  .sort_index()
  .to_frame(name="count")
)

# Convert each count into a percentage of valid actual price records.
price_range_summary["percentage"] = (
  price_range_summary["count"] / len(valid_prices) * 100
)

# Display the price range summary table.
price_range_summary

In [ ]:
# Check that the mutually exclusive price ranges add up to 100%.
price_range_summary["percentage"].sum()

### Finding

Most hourly electricity prices fall within a moderate range.

About 62% of valid records are between 25 and 100 $/MWh. This suggests that most hours are not extreme, even though the market can still experience sharp price spikes.

The dataset also shows meaningful low-price opportunities. About 22% of hours are at or below 25 $/MWh, including about 3% of hours with a zero price. These periods may be useful for flexible household activities such as laundry, dishwashing, or electric vehicle charging.

High-price periods are also frequent enough to matter. About 16% of hours are above 100 $/MWh, and about 4% are above 500 $/MWh. This supports the need for decision support that helps users avoid expensive hours, not just find cheap ones.

These price ranges are exploratory. They describe historical market behavior, but they do not define the final `recommended`, `acceptable`, or `avoid` thresholds yet.

## 5. When do price spikes occur?

This question identifies when unusually expensive hours happen. For EDA, a spike is defined as an actual price above the 95th percentile.

In [ ]:
# Use the 95th percentile as an exploratory spike threshold.
spike_threshold = valid_prices.quantile(0.95)

# Create a Boolean column that marks whether each row is an EDA price spike.
df["is_price_spike_eda"] = df["actual_price"] > spike_threshold

# Count how many hours are above the exploratory spike threshold.
spike_count = int(df["is_price_spike_eda"].sum())

# Calculate the percentage of valid actual price records that are spike hours.
spike_percentage = spike_count / df["actual_price"].notna().sum() * 100

# Print the exploratory spike threshold.
print(f"EDA spike threshold: {spike_threshold:.2f} $/MWh")

# Print the number of spike hours.
print(f"Spike hours: {spike_count:,}")

# Print the percentage of valid records that are spike hours.
print(f"Spike percentage: {spike_percentage:.2f}%")

In [ ]:
# Extract the local calendar year for each row.
df["year"] = df["datetime_local_time"].dt.year

# Extract the local calendar month for each row.
df["month"] = df["datetime_local_time"].dt.month

# Extract the local hour of day for each row.
df["hour"] = df["datetime_local_time"].dt.hour

In [ ]:
# Count exploratory spike hours by year.
spikes_by_year = (
  df.groupby("year")["is_price_spike_eda"]
  .sum()
  .reset_index(name="spike_hours")
)

# Count exploratory spike hours by month.
spikes_by_month = (
  df.groupby("month")["is_price_spike_eda"]
  .sum()
  .reset_index(name="spike_hours")
)

# Count exploratory spike hours by local hour of day.
spikes_by_hour = (
  df.groupby("hour")["is_price_spike_eda"]
  .sum()
  .reset_index(name="spike_hours")
)

# Create one figure with three charts side by side.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Add one overall title for the group of charts.
fig.suptitle("EDA Price Spike Patterns", fontsize=16)

# Plot spike hours by year.
axes[0].bar(spikes_by_year["year"], spikes_by_year["spike_hours"])
axes[0].set_title("By Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Spike hours")

# Plot spike hours by month.
axes[1].bar(spikes_by_month["month"], spikes_by_month["spike_hours"])
axes[1].set_title("By Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Spike hours")
axes[1].set_xticks(range(1, 13))

# Plot spike hours by local hour of day.
axes[2].bar(spikes_by_hour["hour"], spikes_by_hour["spike_hours"])
axes[2].set_title("By Hour of Day")
axes[2].set_xlabel("Hour of day")
axes[2].set_ylabel("Spike hours")
axes[2].set_xticks(range(0, 24))

# Adjust spacing so labels and titles do not overlap.
plt.tight_layout()

# Display the charts.
plt.show()

### Finding

Price spikes show clear time-based patterns.

Spike hours are not evenly distributed across the dataset. They are concentrated more heavily in some years, especially 2022 and 2023. This suggests that electricity price volatility changes over time and that the model may need to handle periods of unusually high market instability.

Spike hours also vary by month. They appear more often in winter, late summer, and early fall, especially in January, July, August, September, and December. This suggests that seasonal conditions may influence spike risk.

The strongest business pattern is by hour of day. Spike hours are much more common in the late afternoon and evening, with the highest concentration around 17:00 to 20:00 local time. This matters for household decision support because many flexible electricity activities can be shifted away from these higher-risk hours.

This spike definition is exploratory. It uses the 95th percentile of actual prices to study historical spike behavior, but it does not define the final spike-risk label for modeling yet.

## 6. What hours of the day tend to be cheaper or more expensive?

This question studies the typical actual price by local hour of day. It helps identify daily usage windows that may be better or worse for flexible household electricity use.

In [ ]:
# Summarize actual prices by local hour of day.
hourly_price_summary = (
  df.dropna(subset=["actual_price"])
  .groupby("hour")["actual_price"]
  .agg(
    mean_price="mean",
    median_price="median",
    p25_price=lambda x: x.quantile(0.25),
    p75_price=lambda x: x.quantile(0.75),
    p95_price=lambda x: x.quantile(0.95),
  )
  .reset_index()
)

# Display the hourly price summary table.
hourly_price_summary

In [ ]:
# Create a new chart area.
plt.figure(figsize=(10, 5))

# Plot the average actual price by hour.
plt.plot(
  hourly_price_summary["hour"],
  hourly_price_summary["mean_price"],
  label="Mean",
)

# Plot the median actual price by hour.
plt.plot(
  hourly_price_summary["hour"],
  hourly_price_summary["median_price"],
  label="Median",
)

# Plot the 95th percentile actual price by hour to show high-price risk.
plt.plot(
  hourly_price_summary["hour"],
  hourly_price_summary["p95_price"],
  label="95th percentile",
)

# Add a clear chart title.
plt.title("Actual Electricity Price by Hour of Day")

# Label the x-axis with local hour of day.
plt.xlabel("Hour of day")

# Label the y-axis with the price unit.
plt.ylabel("Actual price (CAD/MWh)")

# Show all 24 hours on the x-axis.
plt.xticks(range(0, 24))

# Show the chart legend.
plt.legend()

# Display the chart.
plt.show()

### Finding

Electricity prices follow a clear daily pattern.

The lowest-risk hours are generally overnight and early morning. Between about 01:00 and 05:00 local time, median prices are lower, average prices are lower, and the 95th percentile is much lower than during the afternoon and evening. This suggests that these hours may be better candidates for flexible household electricity use.

The highest-risk period appears between 16:00 and 20:00 local time. Median prices during this period are only moderately higher than the rest of the day, but the mean and 95th percentile rise sharply. This means evening hours are not always expensive, but they have a much higher chance of extreme prices.

For the business problem, this pattern matters because households need simple guidance, not only a raw price forecast. The data suggests that overnight usage may often be safer, while late afternoon and evening usage should be treated with more caution.

These findings are exploratory. They describe historical price behavior by hour of day, but they do not define final `recommended`, `acceptable`, or `avoid` rules yet.

## 7. What day-of-week patterns exist?

This question studies whether actual electricity prices change by local day of week. It helps identify whether weekdays and weekends have different price behavior.

In [ ]:
# Extract the local day name for each hourly record.
df["day_of_week"] = df["datetime_local_time"].dt.day_name()

# Define the natural order of the week so the output is not sorted alphabetically.
day_order = [
  "Monday",
  "Tuesday",
  "Wednesday",
  "Thursday",
  "Friday",
  "Saturday",
  "Sunday",
]

In [ ]:
# Summarize actual prices by local day of week.
day_of_week_summary = (
  df.dropna(subset=["actual_price"])
  .groupby("day_of_week")["actual_price"]
  .agg(
    mean_price="mean",
    median_price="median",
    p25_price=lambda x: x.quantile(0.25),
    p75_price=lambda x: x.quantile(0.75),
    p95_price=lambda x: x.quantile(0.95),
  )
  .reindex(day_order)
  .reset_index()
)

# Display the day-of-week price summary table.
day_of_week_summary

In [ ]:
# Create a new chart area.
plt.figure(figsize=(10, 5))

# Plot the average actual price by day of week.
plt.plot(
  day_of_week_summary["day_of_week"],
  day_of_week_summary["mean_price"],
  label="Mean",
)

# Plot the median actual price by day of week.
plt.plot(
  day_of_week_summary["day_of_week"],
  day_of_week_summary["median_price"],
  label="Median",
)

# Plot the 95th percentile actual price by day of week to show high-price risk.
plt.plot(
  day_of_week_summary["day_of_week"],
  day_of_week_summary["p95_price"],
  label="95th percentile",
)

# Add a clear chart title.
plt.title("Actual Electricity Price by Day of Week")

# Label the x-axis with the local day of week.
plt.xlabel("Day of week")

# Label the y-axis with the price unit.
plt.ylabel("Actual price (CAD/MWh)")

# Rotate the day labels so they are easier to read.
plt.xticks(rotation=45)

# Show the chart legend.
plt.legend()

# Display the chart.
plt.show()

### Finding

Electricity prices show a clear weekday and weekend pattern.

Median prices are relatively stable across the week, staying close to 38 to 43 CAD/MWh. This suggests that the typical hourly price does not change dramatically by day of week.

The main difference appears in the mean and 95th percentile. Monday through Thursday have higher average prices and much higher 95th percentile values than Saturday and Sunday. This means weekdays are not always expensive, but they carry a higher risk of extreme prices.

Weekend prices appear less risky. Saturday and Sunday have lower average prices and much lower 95th percentile values, which suggests fewer high-price events on weekends.

For household decision support, day of week may be a useful feature. Weekend hours may often be safer for flexible electricity use, while weekday hours may need more caution, especially when combined with hour-of-day patterns.

These findings are exploratory. They describe historical day-of-week behavior but do not define final recommendation rules yet.

## 8. What monthly or seasonal patterns exist?

This question studies whether actual electricity prices change by local month and season. It helps identify longer-term patterns that may affect price forecasting and spike-risk modeling.

In [ ]:
# Extract the local month number for each hourly record.
df["month"] = df["datetime_local_time"].dt.month

# Define a simple function to map each month to a season.
def get_season(month: int) -> str:
  # Winter includes December, January, and February.
  if month in [12, 1, 2]:
    return "Winter"

  # Spring includes March, April, and May.
  if month in [3, 4, 5]:
    return "Spring"

  # Summer includes June, July, and August.
  if month in [6, 7, 8]:
    return "Summer"

  # Fall includes September, October, and November.
  return "Fall"


# Create a season column from the local month.
df["season"] = df["month"].apply(get_season)

# Define the natural order of seasons for readable output.
season_order = ["Winter", "Spring", "Summer", "Fall"]

In [ ]:
# Summarize actual prices by local month.
monthly_price_summary = (
    df.dropna(subset=["actual_price"])
    .groupby("month")["actual_price"]
    .agg(
      mean_price="mean",
      median_price="median",
      p25_price=lambda x: x.quantile(0.25),
      p75_price=lambda x: x.quantile(0.75),
      p95_price=lambda x: x.quantile(0.95),
    )
    .reset_index()
)

# Display the monthly price summary table.
monthly_price_summary

In [ ]:
# Summarize actual prices by season.
seasonal_price_summary = (
  df.dropna(subset=["actual_price"])
  .groupby("season")["actual_price"]
  .agg(
    mean_price="mean",
    median_price="median",
    p25_price=lambda x: x.quantile(0.25),
    p75_price=lambda x: x.quantile(0.75),
    p95_price=lambda x: x.quantile(0.95),
  )
  .reindex(season_order)
  .reset_index()
)

# Display the seasonal price summary table.
seasonal_price_summary

In [ ]:
# Create one figure with two charts side by side.
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Add one overall title for the group of charts.
fig.suptitle("Actual Electricity Price by Month and Season", fontsize=16)

# Plot monthly average, median, and 95th percentile prices.
axes[0].plot(monthly_price_summary["month"], monthly_price_summary["mean_price"], label="Mean")
axes[0].plot(monthly_price_summary["month"], monthly_price_summary["median_price"], label="Median")
axes[0].plot(monthly_price_summary["month"], monthly_price_summary["p95_price"], label="95th percentile")
axes[0].set_title("By Month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Actual price (CAD/MWh)")
axes[0].set_xticks(range(1, 13))
axes[0].legend()

# Plot seasonal average, median, and 95th percentile prices.
axes[1].plot(seasonal_price_summary["season"], seasonal_price_summary["mean_price"], label="Mean")
axes[1].plot(seasonal_price_summary["season"], seasonal_price_summary["median_price"], label="Median")
axes[1].plot(seasonal_price_summary["season"], seasonal_price_summary["p95_price"], label="95th percentile")
axes[1].set_title("By Season")
axes[1].set_xlabel("Season")
axes[1].set_ylabel("Actual price (CAD/MWh)")
axes[1].legend()

# Adjust spacing so labels and titles do not overlap.
plt.tight_layout()

# Display the charts.
plt.show()

### Finding

Electricity prices show clear monthly and seasonal patterns.

The typical price does not change dramatically across the year. Median prices stay mostly between about 33 and 49 $/MWh. This suggests that the normal hourly price remains relatively stable by month.

The main difference appears in high-price risk. The 95th percentile changes much more than the median, which means some months have a much higher chance of extreme prices. August has the highest 95th percentile, followed by September and December. January and July also show elevated risk.

At the season level, summer has the highest 95th percentile, while spring has the lowest. This suggests that seasonal conditions may help explain when price spikes are more likely.

For modeling, month and season may be useful features. They may help the model capture longer-term price patterns and spike-risk behavior.

These findings are exploratory. They describe historical monthly and seasonal behavior, but they do not define final recommendation rules yet.

## 9. How does the AESO forecast compare with actual prices?

This question studies how close the AESO forecast price is to the actual pool price. It helps decide whether `forecast_price` may be useful as a modeling feature.

In [ ]:
# Keep only rows where both actual and forecast prices are available.
forecast_comparison = df.dropna(subset=["actual_price", "forecast_price"]).copy()


# Calculate the forecast error for each hourly record.
forecast_comparison["forecast_error"] = (
  forecast_comparison["forecast_price"] - forecast_comparison["actual_price"]
)

# Calculate the absolute forecast error for each hourly record.
forecast_comparison["absolute_forecast_error"] = (
  forecast_comparison["forecast_error"].abs()
)

# Display the first rows to confirm the new error columns.
forecast_comparison[
  [
    "datetime_local_time",
    "actual_price",
    "forecast_price",
    "forecast_error",
    "absolute_forecast_error",
  ]
].head()

In [ ]:
# Calculate the mean forecast error to check average forecast bias.
mean_error = forecast_comparison["forecast_error"].mean()

# Calculate the mean absolute error to measure average forecast miss size.
mean_absolute_error = forecast_comparison["absolute_forecast_error"].mean()

# Calculate the median absolute error to measure the typical forecast miss size.
median_absolute_error = forecast_comparison["absolute_forecast_error"].median()

# Calculate correlation between forecast price and actual price.
forecast_actual_correlation = forecast_comparison["forecast_price"].corr(
    forecast_comparison["actual_price"]
)

# Print the key forecast comparison metrics.
print(f"Mean forecast error: {mean_error:.2f} CAD/MWh")
print(f"Mean absolute forecast error: {mean_absolute_error:.2f} CAD/MWh")
print(f"Median absolute forecast error: {median_absolute_error:.2f} CAD/MWh")
print(f"Forecast vs actual correlation: {forecast_actual_correlation:.3f}")

In [ ]:
# Create a new chart area.
plt.figure(figsize=(8, 6))

# Plot actual price against forecast price.
plt.scatter(
    forecast_comparison["forecast_price"],
    forecast_comparison["actual_price"],
    alpha=0.2,
)

# Create the reference range for a perfect forecast line.
max_price = max(
    forecast_comparison["forecast_price"].max(),
    forecast_comparison["actual_price"].max(),
)

# Plot the perfect forecast line where forecast price equals actual price.
plt.plot([0, max_price], [0, max_price], linestyle="--", label="Perfect forecast", color="red")

# Add a clear chart title.
plt.title("AESO Forecast Price vs Actual Price")

# Label the x-axis with forecast price.
plt.xlabel("Forecast price (CAD/MWh)")

# Label the y-axis with actual price.
plt.ylabel("Actual price (CAD/MWh)")

# Show the chart legend.
plt.legend()

# Display the chart.
plt.show()

In [ ]:
# Create a new chart area.
plt.figure(figsize=(10, 5))

# Plot the distribution of absolute forecast errors.
plt.hist(forecast_comparison["absolute_forecast_error"], bins=100)

# Add a clear chart title.
plt.title("Distribution of AESO Absolute Forecast Errors")

# Label the x-axis with absolute forecast error.
plt.xlabel("Absolute forecast error ($/MWh)")

# Label the y-axis with the number of hourly records.
plt.ylabel("Number of hours")

# Display the chart.
plt.show()

### Finding

The AESO forecast price is strongly related to the actual pool price.

The correlation between forecast price and actual price is about 0.91, which shows a strong positive relationship. This means `forecast_price` is likely to be a useful feature for price prediction.

Most forecast errors are small. The median absolute forecast error is about 2.44 CAD/MWh, which means that for a typical hour, the forecast is close to the actual price.

However, the mean absolute forecast error is much higher at about 22.07 CAD/MWh. This difference suggests that some hours have large forecast misses, especially during more volatile market conditions.

For the business problem, this means the AESO forecast can help guide predictions, but the model should not rely on it alone. The project still needs additional time-based features and spike-risk modeling to handle unusual high-price periods.

These findings are exploratory. They show that `forecast_price` is useful, but they do not prove that it is sufficient for final recommendation logic.

## 10. What should be studied before defining recommendation thresholds?

This question summarizes what the EDA shows before defining final `recommended`, `acceptable`, or `avoid` thresholds.

The goal is not to create final rules yet. The goal is to identify which patterns should guide future recommendation logic.

In [ ]:
# Create a simple summary of the main EDA signals that matter for future recommendation logic.
recommendation_inputs = pd.DataFrame(
  [
    {
      "eda_signal": "Price distribution",
      "finding": "Actual prices are strongly right-skewed.",
      "why_it_matters": "The system must handle both normal prices and rare expensive spikes.",
    },
    {
      "eda_signal": "Low-price hours",
      "finding": "About 22% of valid hours are at or below 25 CAD/MWh.",
      "why_it_matters": "Low-price periods may support future recommended usage windows.",
    },
    {
      "eda_signal": "High-price hours",
      "finding": "About 16% of valid hours are above 100 CAD/MWh.",
      "why_it_matters": "High-price periods are common enough to justify avoid warnings.",
    },
    {
      "eda_signal": "Spike behavior",
      "finding": "Spike hours are more common in some years, seasons, months, and evening hours.",
      "why_it_matters": "Spike risk should be modeled, not treated as random noise.",
    },
    {
      "eda_signal": "Hour-of-day pattern",
      "finding": "Overnight hours are generally lower-risk, while 16:00 to 20:00 is higher-risk.",
      "why_it_matters": "Hour of day should be considered for recommendation logic.",
    },
    {
      "eda_signal": "Day-of-week pattern",
      "finding": "Weekdays carry higher high-price risk than weekends.",
      "why_it_matters": "Day of week may help identify safer flexible usage periods.",
    },
    {
      "eda_signal": "Seasonal pattern",
      "finding": "Summer has the highest seasonal 95th percentile, while spring has the lowest.",
      "why_it_matters": "Seasonal features may help explain changing spike risk.",
    },
    {
      "eda_signal": "AESO forecast",
      "finding": "Forecast price is strongly correlated with actual price.",
      "why_it_matters": "Forecast price is likely useful, but not sufficient by itself.",
    },
    {
      "eda_signal": "Load data availability",
      "finding": "`alberta_internal_load` is available in the historical CSV but missing for recent API records.",
      "why_it_matters": "The first modeling dataset should not rely on this feature unless a recent AIL source is added.",
    },
  ]
)

# Display the recommendation input summary.
recommendation_inputs

### Finding

The EDA shows that final recommendation thresholds should not be chosen arbitrarily.

Several patterns should guide the future decision layer. Actual prices are strongly right-skewed, which means the project must handle both normal hourly prices and rare expensive spike periods. Low-price hours exist often enough to support future `recommended` windows, while high-price hours are frequent enough to justify future `avoid` warnings.

Time-based patterns are also important. Hour of day, day of week, month, and season all show useful relationships with price level or spike risk. These patterns should be considered during feature engineering and modeling.

The AESO forecast price is strongly related to the actual price, so it is likely to be a useful modeling feature. However, forecast errors can be large during volatile periods, so the final recommendation logic should not rely on forecast price alone.

The next step is feature engineering. Final `recommended`, `acceptable`, and `avoid` thresholds should be defined later, after the project has tested baseline models, spike-risk classification, and model evaluation results.